# 01 · Dataset preparation and audit (Sprint 1)

SafeRoute uses a **two-stage cascade** (proposal Section 5.2.2):

| Stage | Task | Dataset | Classes |
|---|---|---|---|
| 1 · Detector (YOLO11n) | *Where* are the traffic signs? | PauloLab *Road Signs* v2 (Roboflow Universe, Public Domain) | 1 (`traffic_sign`) |
| 2 · Classifier (baseline CNN) | *Which* sign is it? | GTSRB, mapped to SafeRoute classes | 10 |

Steps covered here (Module 2 deck): **gather → inspect → clean → preprocess → split → verify**.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import matplotlib.pyplot as plt

# Works whether the notebook is opened from the repo root or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
print("project root:", ROOT)

## 1 · Detector dataset: import summary
`scripts/import_roboflow.py` removed Roboflow augmentation duplicates, converted labels to class 0, and re-split **70/15/15 by physical sign** so photos of the same sign never appear in both train and test.

In [ ]:
stats = json.loads((ROOT / "data/processed/signs_v0/import_stats.json").read_text())
for k, v in stats.items():
    print(f"{k:15s} {v}")

## 2 · Inspect and verify: audit the detector dataset
Checks for corrupted images, missing/empty labels, invalid class IDs, out-of-range boxes and tiny boxes.

In [ ]:
import audit_dataset

rep = audit_dataset.audit(ROOT / "data/processed/signs_v0", num_classes=1)
for split, s in rep["splits"].items():
    print(f"{split:5s} images={s['images']:4d}  instances={s['instances']:4d}  background={s['background_images']}")
print(f"\nerrors={len(rep['errors'])}  warnings={len(rep['warnings'])}")
for w in rep["warnings"][:5]:
    print("  warning:", w)

## 3 · Example labeled frame

In [ ]:
from PIL import Image, ImageDraw

img_dir = ROOT / "data/processed/signs_v0/images/train"
lbl_dir = ROOT / "data/processed/signs_v0/labels/train"
for img_path in sorted(img_dir.iterdir()):
    lines = [l for l in (lbl_dir / f"{img_path.stem}.txt").read_text().splitlines() if l.strip()]
    if lines:
        break

im = Image.open(img_path).convert("RGB")
W, H = im.size
draw = ImageDraw.Draw(im)
for ln in lines:
    _, xc, yc, w, h = map(float, ln.split())
    draw.rectangle([(xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H], outline="lime", width=4)
plt.figure(figsize=(9, 6)); plt.imshow(im); plt.axis("off")
plt.title(f"{img_path.name}  ({len(lines)} sign(s))"); plt.show()

## 4 · Classifier dataset: class balance (verify BEFORE training)
`scripts/import_gtsrb.py` kept the 10 GTSRB classes that match SafeRoute and held out **whole GTSRB tracks** (≈30 frames of one physical sign) for validation. Imbalance is handled during training by `WeightedRandomSampler`.

In [ ]:
counts = json.loads((ROOT / "data/processed/crops/crop_counts.json").read_text())
classes = sorted(counts["train"])
x, w = np.arange(len(classes)), 0.27
fig, ax = plt.subplots(figsize=(11, 4))
for i, split in enumerate(["train", "val", "test"]):
    ax.bar(x + (i - 1) * w, [counts[split].get(c, 0) for c in classes], w, label=split)
ax.set_xticks(x); ax.set_xticklabels(classes, rotation=40, ha="right")
ax.set_ylabel("crops"); ax.set_title("Classifier dataset per class and split"); ax.legend()
plt.tight_layout(); plt.show()
for split in ["train", "val", "test"]:
    print(f"{split:5s} total={sum(counts[split].values())}")

## 5 · Example crop per class (preprocessing preview: resized to 64×64 for the CNN)

In [ ]:
crop_root = ROOT / "data/processed/crops/train"
fig, axes = plt.subplots(1, len(classes), figsize=(16, 2.4))
for ax, c in zip(axes, classes):
    sample = next((crop_root / c).iterdir())
    ax.imshow(Image.open(sample).convert("RGB").resize((64, 64))); ax.set_title(c, fontsize=7); ax.axis("off")
plt.tight_layout(); plt.show()